# Assessing Generalization in Superconducting Critical Temperature Prediction Across Chemical Systems

This notebook predicts the critical temperature (Tc) of superconductors from composition-derived features using classical machine learning (Linear Regression, Random Forest, XGBoost) and deep learning (a PyTorch multilayer perceptron). It then tests how well these models generalize to chemical families they have never seen.

**Dataset:** UCI Superconductivity Data (Hamidieh, 2018): 21,263 superconductors, 81 features, measured Tc.

**Runtime:** Runtime → Change runtime type → T4 GPU (optional; speeds up the neural network).

## 1. Setup and data download

In [ ]:
import os
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)

if not os.path.exists("train.csv"):
    !wget -q "https://archive.ics.uci.edu/static/public/464/superconductivty+data.zip" -O sc.zip
    !unzip -o -q sc.zip
# If the download fails, get train.csv and unique_m.csv from Kaggle
# ("superconductivity data UCI") and upload them with the folder icon on the left.
!ls *.csv

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEED = 42
np.random.seed(SEED)

df = pd.read_csv("train.csv")        # 81 features + critical_temp
um = pd.read_csv("unique_m.csv")     # element counts + critical_temp + material (same row order)
assert len(df) == len(um) and np.allclose(df["critical_temp"], um["critical_temp"])
print(df.shape, "| missing values:", df.isna().sum().sum())
df["critical_temp"].describe()

## 2. Exploratory analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(df["critical_temp"], bins=60)
ax[0].set(xlabel="Tc (K)", ylabel="Count", title="Distribution of Tc")
top = df.corr()["critical_temp"].drop("critical_temp").abs().sort_values(ascending=False).head(10)
ax[1].barh(top.index[::-1], top.values[::-1])
ax[1].set(xlabel="|Pearson r| with Tc", title="Top 10 correlated features")
plt.tight_layout(); plt.savefig("figures/eda.png", dpi=150); plt.show()

## 3. Honest train/test split

Some formulas appear more than once. A plain random split can put the same compound in both sets and inflate scores, so the split is grouped by material formula.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

feat_names = df.drop(columns="critical_temp").columns
X = df[feat_names].values
y = df["critical_temp"].values
groups = um["material"].values
print("Unique formulas:", len(set(groups)), "of", len(groups), "rows")

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr, te = next(gss.split(X, y, groups))
X_train, X_test, y_train, y_test = X[tr], X[te], y[tr], y[te]
print("Train:", X_train.shape, " Test:", X_test.shape)

## 4. Classical machine learning models

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

records = []
def evaluate(model, split, y_true, y_pred):
    r = dict(model=model, split=split,
             RMSE=np.sqrt(mean_squared_error(y_true, y_pred)),
             MAE=mean_absolute_error(y_true, y_pred),
             R2=r2_score(y_true, y_pred))
    records.append(r)
    print(f"{model:14s} | {split:24s} | RMSE={r['RMSE']:6.2f} K  MAE={r['MAE']:6.2f} K  R2={r['R2']:.3f}")
    return r

def make_models():
    return {
        "Linear": LinearRegression(),
        "RandomForest": RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=SEED),
        "XGBoost": XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=8,
                                subsample=0.8, colsample_bytree=0.6, random_state=SEED),
    }

models = make_models()
for name, m in models.items():
    m.fit(X_train, y_train)
    evaluate(name, "grouped random 80/20", y_test, m.predict(X_test))

## 5. Deep learning model (PyTorch MLP)

In [ ]:
import torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)

class TcMLP:
    """Small wrapper so the MLP can be fit/predicted like a scikit-learn model."""
    def __init__(self, epochs=300, patience=20, lr=1e-3, batch_size=128, verbose=False):
        self.epochs, self.patience, self.lr, self.bs, self.verbose = epochs, patience, lr, batch_size, verbose

    def _T(self, a):
        return torch.tensor(a, dtype=torch.float32, device=device)

    def _pred_scaled(self, Xs):
        self.net.eval()
        with torch.no_grad():
            return self.sy.inverse_transform(self.net(self._T(Xs)).cpu().numpy()).ravel()

    def fit(self, X, y):
        Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.1, random_state=SEED)
        self.sx, self.sy = StandardScaler().fit(Xtr), StandardScaler().fit(ytr.reshape(-1, 1))
        Xtr_s, Xval_s = self.sx.transform(Xtr), self.sx.transform(Xval)
        ytr_s = self.sy.transform(ytr.reshape(-1, 1))
        d = X.shape[1]
        self.net = nn.Sequential(
            nn.Linear(d, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1)).to(device)
        opt = torch.optim.Adam(self.net.parameters(), lr=self.lr, weight_decay=1e-5)
        loss_fn = nn.MSELoss()
        loader = torch.utils.data.DataLoader(
            torch.utils.data.TensorDataset(self._T(Xtr_s), self._T(ytr_s)),
            batch_size=self.bs, shuffle=True)
        best, wait, best_state = np.inf, 0, None
        self.history = []
        for epoch in range(self.epochs):
            self.net.train()
            for xb, yb in loader:
                opt.zero_grad(); loss = loss_fn(self.net(xb), yb); loss.backward(); opt.step()
            val_rmse = np.sqrt(mean_squared_error(yval, self._pred_scaled(Xval_s)))
            self.history.append(val_rmse)
            if val_rmse < best:
                best, wait = val_rmse, 0
                best_state = {k: v.detach().clone() for k, v in self.net.state_dict().items()}
            else:
                wait += 1
                if wait > self.patience:
                    break
            if self.verbose and epoch % 10 == 0:
                print(f"epoch {epoch:3d}  val RMSE {val_rmse:6.2f} K")
        self.net.load_state_dict(best_state)
        return self

    def predict(self, X):
        return self._pred_scaled(self.sx.transform(X))

mlp = TcMLP(verbose=True).fit(X_train, y_train)
models["MLP (DL)"] = mlp
evaluate("MLP (DL)", "grouped random 80/20", y_test, mlp.predict(X_test))

plt.plot(mlp.history); plt.xlabel("Epoch"); plt.ylabel("Validation RMSE (K)")
plt.title("MLP training curve"); plt.savefig("figures/mlp_training.png", dpi=150); plt.show()

## 6. Model comparison

In [ ]:
res = pd.DataFrame(records)
base = res[res.split == "grouped random 80/20"]
plt.bar(base.model, base.RMSE); plt.ylabel("Test RMSE (K)"); plt.title("Model comparison")
plt.savefig("figures/model_comparison.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, name in zip(axes, ["XGBoost", "MLP (DL)"]):
    p = models[name].predict(X_test)
    ax.scatter(y_test, p, s=4, alpha=0.4)
    ax.plot([0, 140], [0, 140], "r--")
    ax.set(xlabel="Measured Tc (K)", ylabel="Predicted Tc (K)", title=name)
plt.tight_layout(); plt.savefig("figures/parity_plots.png", dpi=150); plt.show()

## 7. Interpretation with SHAP

In [ ]:
import shap
explainer = shap.TreeExplainer(models["XGBoost"])
sample = X_test[:2000]
sv = explainer.shap_values(sample)
shap.summary_plot(sv, sample, feature_names=list(feat_names), max_display=15, show=False)
plt.tight_layout(); plt.savefig("figures/shap_summary.png", dpi=150, bbox_inches="tight"); plt.show()

## 8. Generalization across chemical systems

Each material is labeled as Cuprate, Iron-based or Other from its composition. Every model is trained on two families and tested on the family it has never seen (leave-one-family-out).

In [ ]:
def family(r):
    if r["Cu"] > 0 and r["O"] > 0:
        return "Cuprate"
    if r["Fe"] > 0 and (r["As"] > 0 or r["Se"] > 0 or r["P"] > 0 or r["S"] > 0 or r["Te"] > 0):
        return "Iron-based"
    return "Other"

fam = um.apply(family, axis=1).values
print(pd.Series(fam).value_counts())

fig, ax = plt.subplots(figsize=(7, 4))
for f in ["Cuprate", "Iron-based", "Other"]:
    ax.hist(y[fam == f], bins=50, alpha=0.5, label=f)
ax.set(xlabel="Tc (K)", ylabel="Count", title="Tc by chemical family"); ax.legend()
plt.savefig("figures/tc_by_family.png", dpi=150); plt.show()

In [ ]:
for held_out in ["Cuprate", "Iron-based", "Other"]:
    tr_idx, te_idx = fam != held_out, fam == held_out
    lofo_models = make_models()
    lofo_models["MLP (DL)"] = TcMLP()
    for name, m in lofo_models.items():
        m.fit(X[tr_idx], y[tr_idx])
        evaluate(name, f"leave out {held_out}", y[te_idx], m.predict(X[te_idx]))

## 9. Results summary

In [ ]:
res = pd.DataFrame(records)
res.to_csv("results/results.csv", index=False)
table = res.pivot(index="model", columns="split", values="RMSE").round(2)
display(table)

table.T.plot(kind="bar", figsize=(10, 4.5))
plt.ylabel("Test RMSE (K)"); plt.title("Random split vs. unseen chemical families")
plt.xticks(rotation=20, ha="right"); plt.tight_layout()
plt.savefig("figures/generalization_rmse.png", dpi=150); plt.show()

## 10. Discussion prompts

- How much does RMSE grow from the random split to each leave-one-family-out split?
- Which model degrades least, and why might tree models fail to extrapolate beyond the Tc range seen in training?
- Do the SHAP top features have a physical interpretation (e.g., thermal conductivity, valence, atomic mass)?
- What does this imply for using ML to screen genuinely new superconductor families?

**Reference:** Hamidieh, K. (2018). A data-driven statistical model for predicting the critical temperature of a superconductor. *Computational Materials Science*, 154, 346–354.